In [2]:
import cv2
import requests

# 打开摄像头
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # 显示摄像头画面
    cv2.imshow('Camera', frame)

    # 按下 'q' 键退出
    if cv2.waitKey(1) & 0xFF == ord('q'):
        # 将图像编码为 JPEG 格式
        _, buffer = cv2.imencode('.jpg', frame)
        image_data = buffer.tobytes()

        # 发送 POST 请求
        files = {'image': ('frame.jpg', image_data, 'image/jpeg')}
        response = requests.post('http://127.0.0.1:5000/predict_facemoe', files=files)

        # 打印预测结果
        print(response.json())

# 释放摄像头并关闭窗口
cap.release()
cv2.destroyAllWindows()

{'predicted_emotion': '快乐'}


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [ ]:
from flask import Flask, request, jsonify
import torch
from PIL import Image
from transformers import AutoImageProcessor, AutoModelForImageClassification
import io

app = Flask(__name__)

# 加载图像处理器和模型
processor = AutoImageProcessor.from_pretrained("D:\huggingface-download\models--dima806--facial_emotions_image_detection")
model = AutoModelForImageClassification.from_pretrained("D:\huggingface-download\models--dima806--facial_emotions_image_detection")

# 定义标签对应的中文情感
labels = {
    0: "悲伤",
    1: "厌恶",
    2: "愤怒",
    3: "中性",
    4: "恐惧",
    5: "惊喜",
    6: "快乐",
}

@app.route('/predict_facemoe', methods=['POST'])
def predict_facemoe():
    if 'image' not in request.files:
        return jsonify({'error': 'No image provided'}), 400

    file = request.files['image']
    image = Image.open(io.BytesIO(file.read()))

    # 预处理图像
    inputs = processor(images=image, return_tensors="pt")

    # 进行推理
    with torch.no_grad():
        outputs = model(**inputs)

    # 获取预测结果
    logits = outputs.logits
    predicted_class = logits.argmax(-1).item()

    # 获取预测的情感
    predicted_emotion = labels[predicted_class]

    return jsonify({'predicted_emotion': predicted_emotion})

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000)

{'predicted_emotion': '悲伤'}
{'predicted_emotion': '快乐'}
{'predicted_emotion': '恐惧'}
{'predicted_emotion': '悲伤'}

In [6]:
import sounddevice as sd
import numpy as np
import requests
import wave

def record_audio(duration, filename):
    print(f"Recording for {duration} seconds...")
    audio = sd.rec(int(duration * 16000), samplerate=16000, channels=1, dtype='int16')
    sd.wait()  # 等待录音完成
    wav_file = wave.open(filename, 'wb')
    wav_file.setnchannels(1)
    wav_file.setsampwidth(2)  # 每个样本的字节数
    wav_file.setframerate(16000)
    wav_file.writeframes(audio.tobytes())
    wav_file.close()
    print(f"Recording saved to {filename}")

# 录制两段音频
record_audio(5, 'speaker1.wav')
record_audio(5, 'speaker2.wav')

# 发送 POST 请求
url = 'http://127.0.0.1:5000/verify_audioid'
files = {
    'speaker1': open('speaker1.wav', 'rb'),
    'speaker2': open('speaker2.wav', 'rb'),
}
response = requests.post(url, files=files)

# 打印预测结果
print(response.json())

Recording for 5 seconds...
Recording saved to speaker1.wav
Recording for 5 seconds...
Recording saved to speaker2.wav
{'score': 0.76595, 'text': 'yes'}


两个不同
Recording for 5 seconds...
Recording saved to speaker1.wav
Recording for 5 seconds...
Recording saved to speaker2.wav
{'score': 0.58129, 'text': 'yes'}

Recording for 5 seconds...
Recording saved to speaker1.wav
Recording for 5 seconds...
Recording saved to speaker2.wav
{'score': 0.49643, 'text': 'yes'}

------------------------------------------------
两个相同
Recording for 5 seconds...
Recording saved to speaker1.wav
Recording for 5 seconds...
Recording saved to speaker2.wav
{'score': 0.76595, 'text': 'yes'}

In [15]:
import sounddevice as sd
import numpy as np
import requests
import wave

def record_audio(duration, filename):
    print(f"Recording for {duration} seconds...")
    audio = sd.rec(int(duration * 16000), samplerate=16000, channels=1, dtype='int16')
    sd.wait()  # 等待录音完成
    with wave.open(filename, 'wb') as wav_file:
        wav_file.setnchannels(1)
        wav_file.setsampwidth(2)  # 每个样本的字节数
        wav_file.setframerate(16000)
        wav_file.writeframes(audio.tobytes())
    print(f"Recording saved to {filename}")

# 录制音频
record_audio(5, 'audio.wav')

# 发送 POST 请求
url = 'http://127.0.0.1:5000/audio2txt'
files = {'audio': open('audio.wav', 'rb')}
response = requests.post(url, files=files)

# 打印转录结果
print(response.json())

{'transcription': [{'key': 'rand_key_2yW4Acq9GFz6Y', 'text': '你好，我是大傻逼，我是刘吉。', 'timestamp': [[1450, 1610], [1610, 1790], [1790, 1890], [1890, 2010], [2010, 2130], [2130, 2290], [2290, 2530], [2650, 2770], [2770, 2870], [2870, 3050], [3050, 3375]]}]}


{'transcription': [{'key': 'rand_key_2yW4Acq9GFz6Y', 'text': '你好，我是大傻逼，我是刘吉。', 'timestamp': [[1450, 1610], [1610, 1790], [1790, 1890], [1890, 2010], [2010, 2130], [2130, 2290], [2290, 2530], [2650, 2770], [2770, 2870], [2870, 3050], [3050, 3375]]}]}

In [18]:
import requests

# 发送 POST 请求
url = 'http://127.0.0.1:5000/classify_text2emo'
text_data = {'text': '拜拜 世界'}
response = requests.post(url, json=text_data)

# 打印返回结果
print(response.json())

text_data = {'text': '你好 世界'}
response = requests.post(url, json=text_data)

# 打印返回结果
print(response.json())

{'results': [{'label': '悲伤', 'score': 0.9980395436286926}]}
{'results': [{'label': '高兴', 'score': 0.6344264149665833}, {'label': '喜好', 'score': 0.33021819591522217}]}


{'results': [{'label': '悲伤', 'score': 0.9980395436286926}]}
{'results': [{'label': '高兴', 'score': 0.6344264149665833}, {'label': '喜好', 'score': 0.33021819591522217}]}

In [26]:
import sounddevice as sd
import numpy as np
import requests
import wave

def record_audio(duration, filename):
    print(f"Recording for {duration} seconds...")
    audio = sd.rec(int(duration * 16000), samplerate=16000, channels=1, dtype='int16')
    sd.wait()  # 等待录音完成
    with wave.open(filename, 'wb') as wav_file:
        wav_file.setnchannels(1)
        wav_file.setsampwidth(2)  # 每个样本的字节数
        wav_file.setframerate(16000)
        wav_file.writeframes(audio.tobytes())
    print(f"Recording saved to {filename}")

# 录制音频
# record_audio(5, 'audio.wav')

# 发送 POST 请求
url = 'http://127.0.0.1:5000/classify_audio2emo'
files = {'audio': open('audio.wav', 'rb')}
response = requests.post(url, files=files)

# 打印预测结果
print(response.json())

{'emotion': {'index': 2, 'label': ['hap'], 'probabilities': [[1.9798623895894707e-07, 7.27307970793678e-12, 0.9999997615814209, 6.425798781961589e-11]], 'score': 0.9999997615814209}}


{'emotion': {'index': 2, 'label': ['hap'], 'probabilities': [[1.9798623895894707e-07, 7.27307970793678e-12, 0.9999997615814209, 6.425798781961589e-11]], 'score': 0.9999997615814209}}

In [ ]:
import cv2
import requests

# 打开摄像头
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # 显示摄像头画面
    cv2.imshow('Camera', frame)

    # 按下 's' 键发送当前帧
    if cv2.waitKey(1) & 0xFF == ord('s'):
        # 将图像编码为 JPEG 格式
        _, img_encoded = cv2.imencode('.jpg', frame)
        image_data = img_encoded.tobytes()

        # 发送 POST 请求
        files = {'image': ('frame.jpg', image_data, 'image/jpeg')}
        response = requests.post('http://127.0.0.1:5000/detect_faceid', files=files)

        # 打印检测结果
        print(response.json())

    # 按下 'q' 键退出
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# 释放摄像头并关闭窗口
cap.release()
cv2.destroyAllWindows()

{'results': [0.8389119505882263, 'lhk (2)', [[311, 151], [355, 142]], 1]}


{'results': [0.7943910956382751, 'lhk (2)', [[374, 137], [428, 137]], 0]}
{'results': [0.8439862728118896, 'lhk (2)', [[348, 135], [399, 127]], 1]}
{'results': [0.8026680946350098, 'lj (4)', [[353, 115], [399, 111]], 1]}